In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Locate your two Parquet files

In [ ]:
import os

drive_path = "/content/drive/MyDrive/ScaleWave/Data"

print("Files and folders in MyDrive:\n")

for item in os.listdir(drive_path):
    print(item)

Import libraries and define file paths

In [ ]:
import os
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

# Google Drive directory
DATA_DIR = "/content/drive/MyDrive/ScaleWave/Data"

# Dataset 1: Non-extreme returns
normal_file = os.path.join(
    DATA_DIR,
    "BTCUSDT_signed_return_between_minus500_and_plus500_20261027_101854_1aefb791.parquet"
)

# Dataset 2: Extreme returns
extreme_file = os.path.join(
    DATA_DIR,
    "BTCUSDT_signed_return_above_500_or_below_minus500_20261027_093028_e1453352.parquet"
)

# Verify file accessibility
for label, path in [
    ("Non-extreme dataset", normal_file),
    ("Extreme dataset", extreme_file)
]:
    print(f"{label}:")
    print(f"  Exists: {os.path.isfile(path)}")
    print(f"  Size: {os.path.getsize(path) / (1024**3):.3f} GB"
          if os.path.isfile(path) else "  File not accessible")
    print()

Inspect row counts and schemas

In [ ]:
# Open Parquet metadata without loading the full datasets
normal_pf = pq.ParquetFile(normal_file)
extreme_pf = pq.ParquetFile(extreme_file)

# Display basic information
for name, pf in [
    ("NON-EXTREME DATASET", normal_pf),
    ("EXTREME DATASET", extreme_pf)
]:
    print("=" * 60)
    print(name)
    print("=" * 60)

    print(f"Rows: {pf.metadata.num_rows:,}")
    print(f"Columns: {len(pf.schema_arrow.names):,}")
    print(f"Row groups: {pf.metadata.num_row_groups:,}")

    print("\nFirst 20 columns:")
    for col in pf.schema_arrow.names[:20]:
        print(f"  {col}")

    print()

Compare column structures

In [ ]:
# Get column names
normal_cols = set(normal_pf.schema_arrow.names)
extreme_cols = set(extreme_pf.schema_arrow.names)

# Compare schemas
common_cols = normal_cols & extreme_cols
normal_only = normal_cols - extreme_cols
extreme_only = extreme_cols - normal_cols

print("COLUMN COMPARISON")
print("=" * 50)

print(f"Common columns: {len(common_cols):,}")
print(f"Only in non-extreme dataset: {len(normal_only):,}")
print(f"Only in extreme dataset: {len(extreme_only):,}")

if normal_only:
    print("\nColumns only in non-extreme dataset:")
    print(sorted(normal_only)[:30])

if extreme_only:
    print("\nColumns only in extreme dataset:")
    print(sorted(extreme_only)[:30])

# Identify potential F4Z and Y columns
f4z_cols = [
    col for col in common_cols
    if col.lower().startswith(("f4z_", "f4z__"))
]

potential_y_cols = [
    col for col in common_cols
    if any(keyword in col.lower()
           for keyword in ["signed_return", "return_bps", "label"])
]

print("\nF4Z FEATURE COLUMNS:", len(f4z_cols))

print("\nPOTENTIAL Y COLUMNS:")
for col in sorted(potential_y_cols)[:30]:
    print(col)

Classify F4Z columns

In [ ]:
# Identify actual F4Z feature values
f4z_value_cols = sorted([
    col for col in common_cols
    if col.lower().startswith("f4z__")
    and col.endswith("__value")
])

# Identify validity columns
f4z_valid_cols = sorted([
    col for col in common_cols
    if col.lower().startswith("f4z__")
    and col.endswith("__valid")
])

# Identify other F4Z metadata columns
f4z_metadata_cols = sorted([
    col for col in common_cols
    if col.lower().startswith("f4z__")
    and not col.endswith(("__value", "__valid"))
])

print("F4Z COLUMN BREAKDOWN")
print("=" * 55)

print(f"Actual F4Z value columns: {len(f4z_value_cols):,}")
print(f"F4Z validity columns: {len(f4z_valid_cols):,}")
print(f"Other F4Z metadata columns: {len(f4z_metadata_cols):,}")

print("\nFirst 15 F4Z value columns:")
for col in f4z_value_cols[:15]:
    print(col)

Inspect Y and dataset metadata

In [ ]:
# Identify non-F4Z columns
other_cols = sorted([
    col for col in common_cols
    if not col.lower().startswith("f4z__")
])

print("NON-F4Z COLUMNS")
print("=" * 55)

for col in other_cols:
    dtype = normal_pf.schema_arrow.field(col).type
    print(f"{col}: {dtype}")

Inspect some actual Y values

Inspect some actual Y values

In [ ]:
# Read only essential columns from the first row group
preview_cols = [
    "decision_time",
    "canonical_symbol",
    "signed_return",
    "label_status"
]

# Read first row group from each file
normal_preview = normal_pf.read_row_group(
    0, columns=preview_cols
).slice(0, 10).to_pandas()

extreme_preview = extreme_pf.read_row_group(
    0, columns=preview_cols
).slice(0, 10).to_pandas()

print("NON-EXTREME DATASET SAMPLE")
display(normal_preview)

print("\nEXTREME DATASET SAMPLE")
display(extreme_preview)

Validate Y and establish the research periods

Validate the 48-hour target

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

# Only load the columns needed for label validation
label_cols = [
    "decision_time",
    "canonical_symbol",
    "signed_return",
    "horizon_minutes",
    "label_status"
]

normal_labels = pq.read_table(
    normal_file,
    columns=label_cols
).to_pandas()

extreme_labels = pq.read_table(
    extreme_file,
    columns=label_cols
).to_pandas()

print("NON-EXTREME DATASET")
print("=" * 55)

print("Rows:", len(normal_labels))
print("Symbols:", normal_labels["canonical_symbol"].unique())
print("Horizons:", normal_labels["horizon_minutes"].unique())
print("Label statuses:")
print(normal_labels["label_status"].value_counts(dropna=False))

print("\nEXTREME DATASET")
print("=" * 55)

print("Rows:", len(extreme_labels))
print("Symbols:", extreme_labels["canonical_symbol"].unique())
print("Horizons:", extreme_labels["horizon_minutes"].unique())
print("Label statuses:")
print(extreme_labels["label_status"].value_counts(dropna=False))

Verify the return categories

In [ ]:
# Extract the return values
normal_y = normal_labels["signed_return"]
extreme_y = extreme_labels["signed_return"]

print("RETURN CATEGORY VALIDATION")
print("=" * 60)

print("\nNON-EXTREME DATASET")
print("Minimum return:", normal_y.min())
print("Maximum return:", normal_y.max())

normal_violations = (
    (normal_y < -500) |
    (normal_y > 500)
).sum()

print("Rows outside [-500, +500]:", normal_violations)

print("\nEXTREME DATASET")
print("Minimum return:", extreme_y.min())
print("Maximum return:", extreme_y.max())

extreme_violations = (
    (extreme_y >= -500) &
    (extreme_y <= 500)
).sum()

print("Rows inside [-500, +500]:", extreme_violations)

print("\nMISSING RETURN VALUES")
print("Non-extreme:", normal_y.isna().sum())
print("Extreme:", extreme_y.isna().sum())

Calculate Y event frequencies

In [ ]:
# Combine the label-only datasets
all_labels = pd.concat(
    [normal_labels, extreme_labels],
    ignore_index=True
)

# Define event categories
positive_extreme = all_labels["signed_return"] > 500

negative_extreme = all_labels["signed_return"] < -500

non_extreme = all_labels["signed_return"].between(
    -500, 500, inclusive="both"
)

# Calculate counts
total_rows = len(all_labels)

summary = pd.DataFrame({
    "Category": [
        "Positive extreme (> +500 bps)",
        "Non-extreme (-500 to +500 bps)",
        "Negative extreme (< -500 bps)"
    ],
    "Rows": [
        positive_extreme.sum(),
        non_extreme.sum(),
        negative_extreme.sum()
    ]
})

summary["Percentage"] = (
    summary["Rows"] / total_rows * 100
)

print("BTC 48-HOUR RETURN DISTRIBUTION")
print("=" * 60)

display(summary)

print(f"\nTotal observations: {total_rows:,}")

print(
    f"Positive extreme base rate: "
    f"{positive_extreme.mean() * 100:.4f}%"
)

Analyze timestamps and historical coverage

Inspect historical coverage

In [ ]:
# Sort label datasets by decision time
normal_labels = normal_labels.sort_values("decision_time")
extreme_labels = extreme_labels.sort_values("decision_time")

print("HISTORICAL DATA COVERAGE")
print("=" * 60)

for name, df in [
    ("NON-EXTREME DATASET", normal_labels),
    ("EXTREME DATASET", extreme_labels)
]:
    print(f"\n{name}")
    print("-" * 50)

    print("Start:", df["decision_time"].min())
    print("End:", df["decision_time"].max())
    print("Total rows:", f"{len(df):,}")

    print(
        "Unique timestamps:",
        f"{df['decision_time'].nunique():,}"
    )

    print(
        "Duplicate timestamps:",
        f"{df['decision_time'].duplicated().sum():,}"
    )

Check timestamp overlap

In [ ]:
# Extract timestamps
normal_times = pd.Index(
    normal_labels["decision_time"]
)

extreme_times = pd.Index(
    extreme_labels["decision_time"]
)

# Check overlap
overlap = normal_times.intersection(extreme_times)

print("TIMESTAMP OVERLAP CHECK")
print("=" * 60)

print("Non-extreme timestamps:", f"{len(normal_times):,}")
print("Extreme timestamps:", f"{len(extreme_times):,}")

print("Overlapping timestamps:", f"{len(overlap):,}")

if len(overlap) == 0:
    print("\nPASS: No overlapping timestamps.")
else:
    print("\nWARNING: Overlapping timestamps detected.")

Analyze timestamp gaps

In [ ]:
# Combine and sort all labels
all_labels = pd.concat(
    [normal_labels, extreme_labels],
    ignore_index=True
)

all_labels = (
    all_labels
    .sort_values("decision_time")
    .reset_index(drop=True)
)

# Calculate differences between consecutive timestamps
time_diffs = all_labels["decision_time"].diff()

print("TIMESTAMP GAP ANALYSIS")
print("=" * 60)

print("Total rows:", f"{len(all_labels):,}")

print("\nMost common timestamp intervals:")
print(time_diffs.value_counts().head(10))

# Count gaps larger than one minute
large_gaps = time_diffs > pd.Timedelta(minutes=1)

print("\nGaps greater than 1 minute:", f"{large_gaps.sum():,}")

if large_gaps.any():
    print("\nLargest gaps:")
    print(time_diffs[large_gaps].sort_values(ascending=False).head(10))

Investigate the missing period

Identify the exact missing period

In [ ]:
# Find timestamp gaps larger than one minute
gap_mask = (
    all_labels["decision_time"].diff()
    > pd.Timedelta(minutes=1)
)

gap_indices = all_labels.index[gap_mask]

print("MISSING PERIOD INVESTIGATION")
print("=" * 60)

for idx in gap_indices:

    previous_time = all_labels.loc[idx - 1, "decision_time"]
    next_time = all_labels.loc[idx, "decision_time"]

    gap_duration = next_time - previous_time

    missing_minutes = int(
        gap_duration / pd.Timedelta(minutes=1)
    ) - 1

    print("Last available timestamp:", previous_time)
    print("Next available timestamp:", next_time)
    print("Gap duration:", gap_duration)
    print("Missing minute observations:", missing_minutes)
    print()

Count positive extreme-return episodes

Identify contiguous positive-event episodes

In [ ]:
# Work with the chronologically sorted label dataset
episodes_df = all_labels[
    ["decision_time", "signed_return"]
].copy()

# Define positive extreme events
episodes_df["positive_event"] = (
    episodes_df["signed_return"] > 500
)

# Identify where a new positive-event episode begins
time_gap = (
    episodes_df["decision_time"].diff()
    != pd.Timedelta(minutes=1)
)

previous_event = (
    episodes_df["positive_event"]
    .shift(fill_value=False)
)

new_episode = (
    episodes_df["positive_event"]
    & (~previous_event | time_gap)
)

# Assign episode IDs
episodes_df["episode_id"] = (
    new_episode.cumsum()
    .where(episodes_df["positive_event"])
)

# Summarize episodes
positive_episodes = (
    episodes_df[episodes_df["positive_event"]]
    .groupby("episode_id")
    .agg(
        start_time=("decision_time", "min"),
        end_time=("decision_time", "max"),
        duration_minutes=("decision_time", "size"),
        max_return_bps=("signed_return", "max"),
        mean_return_bps=("signed_return", "mean")
    )
    .reset_index()
)

print("POSITIVE EXTREME EPISODE ANALYSIS")
print("=" * 60)

print("Positive event rows:", f"{episodes_df['positive_event'].sum():,}")
print("Contiguous episodes:", f"{len(positive_episodes):,}")

print("\nEpisode duration statistics (minutes):")
display(positive_episodes["duration_minutes"].describe())

print("\nFirst 10 episodes:")
display(positive_episodes.head(10))

Define discovery and validation periods

In [ ]:
import pandas as pd

# Chronological boundaries (UTC)
DISCOVERY_END = pd.Timestamp("2024-07-01 00:00:00", tz="UTC")
VALIDATION_START = pd.Timestamp("2024-07-03 00:00:00", tz="UTC")

# Create research-period labels
all_labels["research_period"] = "PURGE"

all_labels.loc[
    all_labels["decision_time"] < DISCOVERY_END,
    "research_period"
] = "DISCOVERY"

all_labels.loc[
    all_labels["decision_time"] >= VALIDATION_START,
    "research_period"
] = "VALIDATION"

# Count observations per period
period_counts = (
    all_labels["research_period"]
    .value_counts()
    .rename_axis("Research Period")
    .reset_index(name="Rows")
)

print("CHRONOLOGICAL RESEARCH SPLIT")
print("=" * 60)

display(period_counts)

print("\nPeriod date ranges:")

for period in ["DISCOVERY", "PURGE", "VALIDATION"]:
    subset = all_labels[
        all_labels["research_period"] == period
    ]

    print(f"\n{period}")
    print("Rows:", f"{len(subset):,}")

    if not subset.empty:
        print("Start:", subset["decision_time"].min())
        print("End:", subset["decision_time"].max())

Check the number of positive events in each period

In [ ]:
# Create event categories
all_labels["event_category"] = "NON_EXTREME"

all_labels.loc[
    all_labels["signed_return"] > 500,
    "event_category"
] = "POSITIVE_EXTREME"

all_labels.loc[
    all_labels["signed_return"] < -500,
    "event_category"
] = "NEGATIVE_EXTREME"

# Count categories within each period
period_summary = pd.crosstab(
    all_labels["research_period"],
    all_labels["event_category"]
)

# Calculate positive-event base rate
period_summary["TOTAL"] = period_summary.sum(axis=1)

period_summary["POSITIVE_EVENT_RATE_%"] = (
    period_summary["POSITIVE_EXTREME"]
    / period_summary["TOTAL"]
    * 100
)

print("EVENT DISTRIBUTION BY RESEARCH PERIOD")
print("=" * 70)

display(period_summary)

F4Z feature validity

Inspect F4Z availability fields

In [ ]:
# Identify F4Z availability timestamps
availability_cols = sorted([
    col for col in common_cols
    if col.startswith("f4z__")
    and col.endswith("__available_after")
])

print("F4Z AVAILABILITY FIELDS")
print("=" * 60)

print(f"Total availability columns: {len(availability_cols)}")

print("\nAvailability columns:")
for col in availability_cols:
    print(col)

Check feature availability

In [ ]:
# Read decision time and feature availability fields
audit_cols = ["decision_time"] + availability_cols

normal_availability = pq.read_table(
    normal_file,
    columns=audit_cols
).to_pandas()

extreme_availability = pq.read_table(
    extreme_file,
    columns=audit_cols
).to_pandas()

# Combine timestamps
availability_df = pd.concat(
    [normal_availability, extreme_availability],
    ignore_index=True
)

# Convert to UTC
availability_df["decision_time"] = pd.to_datetime(
    availability_df["decision_time"],
    utc=True
)

results = []

for col in availability_cols:

    available_time = pd.to_datetime(
        availability_df[col],
        utc=True,
        errors="coerce"
    )

    decision_time = availability_df["decision_time"]

    valid_timestamp = available_time.notna()

    future_available = (
        valid_timestamp
        & (available_time > decision_time)
    )

    results.append({
        "Feature Family": col,
        "Missing Availability": (~valid_timestamp).sum(),
        "Available After Decision": future_available.sum(),
        "Availability Rate (%)": (
            (valid_timestamp & ~future_available).mean() * 100
        )
    })

availability_summary = pd.DataFrame(results)

print("F4Z AVAILABILITY AUDIT")
print("=" * 70)

display(availability_summary)

Audit all 522 F4Z features

Audit F4Z feature validity

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd
import gc

# Number of features processed together
FEATURE_BATCH_SIZE = 25

# Rows processed per batch
ROW_BATCH_SIZE = 16384

value_cols = sorted(f4z_value_cols)

validity_results = []

for dataset_name, pf in [
    ("NON_EXTREME", normal_pf),
    ("EXTREME", extreme_pf)
]:

    print(f"\nProcessing {dataset_name}...")

    # Process features in groups
    for start in range(0, len(value_cols), FEATURE_BATCH_SIZE):

        feature_group = value_cols[
            start:start + FEATURE_BATCH_SIZE
        ]

        # Corresponding validity columns
        valid_group = [
            col[:-7] + "__valid"
            for col in feature_group
        ]

        selected_cols = feature_group + valid_group

        # Initialize counters
        counters = {
            col: {
                "total": 0,
                "valid": 0,
                "missing": 0,
                "usable": 0
            }
            for col in feature_group
        }

        # Read selected columns in small batches
        for batch in pf.iter_batches(
            batch_size=ROW_BATCH_SIZE,
            columns=selected_cols,
            use_threads=False
        ):

            # Avoid converting to Pandas
            for value_col, valid_col in zip(
                feature_group, valid_group
            ):

                values = batch.column(
                    batch.schema.get_field_index(value_col)
                )

                flags = batch.column(
                    batch.schema.get_field_index(valid_col)
                )

                n = len(values)

                valid_mask = pc.fill_null(
                    flags, False
                )

                usable_mask = pc.and_(
                    valid_mask,
                    pc.is_valid(values)
                )

                counters[value_col]["total"] += n

                counters[value_col]["valid"] += (
                    pc.sum(
                        pc.cast(valid_mask, pa.int64())
                    ).as_py() or 0
                )

                counters[value_col]["missing"] += (
                    values.null_count
                )

                counters[value_col]["usable"] += (
                    pc.sum(
                        pc.cast(usable_mask, pa.int64())
                    ).as_py() or 0
                )

            del batch

        # Store summary results
        for col, counts in counters.items():

            total = counts["total"]

            validity_results.append({
                "Dataset": dataset_name,
                "Feature": col,
                "Total Rows": total,
                "Valid Rows": counts["valid"],
                "Missing Values": counts["missing"],
                "Usable Rows": counts["usable"],
                "Usable Rate (%)": (
                    counts["usable"] / total * 100
                    if total else 0
                )
            })

        print(
            f"Completed features: "
            f"{min(start + FEATURE_BATCH_SIZE, len(value_cols))}"
            f"/{len(value_cols)}"
        )

        gc.collect()

# Final result
validity_df = pd.DataFrame(validity_results)

print("\nFEATURE VALIDITY AUDIT COMPLETE")

display(validity_df.head(10))

Summarize feature coverage

In [ ]:
coverage_summary = (
    validity_df.groupby("Dataset")
    .agg(
        Features=("Feature", "count"),
        Average_Usable_Rate=("Usable Rate (%)", "mean"),
        Minimum_Usable_Rate=("Usable Rate (%)", "min"),
        Maximum_Usable_Rate=("Usable Rate (%)", "max")
    )
    .reset_index()
)

print("F4Z FEATURE COVERAGE SUMMARY")
display(coverage_summary)

low_coverage = validity_df[
    validity_df["Usable Rate (%)"] < 80
].sort_values("Usable Rate (%)")

print(
    "\nFeatures below 80% coverage:",
    low_coverage["Feature"].nunique()
)

display(low_coverage.head(20))

Compare feature coverage

In [ ]:
import pandas as pd
import numpy as np

# Reshape existing validity results
coverage_comparison = validity_df.pivot(
    index="Feature",
    columns="Dataset",
    values="Usable Rate (%)"
).reset_index()

# Rename columns
coverage_comparison.columns.name = None

coverage_comparison = coverage_comparison.rename(
    columns={
        "NON_EXTREME": "Non_Extreme_Coverage",
        "EXTREME": "Extreme_Coverage"
    }
)

# Calculate difference
coverage_comparison["Coverage_Difference"] = (
    coverage_comparison["Extreme_Coverage"]
    - coverage_comparison["Non_Extreme_Coverage"]
)

# Extract feature family
coverage_comparison["Family"] = (
    coverage_comparison["Feature"]
    .str.split("__")
    .str[1]
)

# Summary
zero_both = (
    (coverage_comparison["Non_Extreme_Coverage"] == 0)
    & (coverage_comparison["Extreme_Coverage"] == 0)
)

high_both = (
    (coverage_comparison["Non_Extreme_Coverage"] >= 90)
    & (coverage_comparison["Extreme_Coverage"] >= 90)
)

moderate_both = (
    (coverage_comparison["Non_Extreme_Coverage"] >= 70)
    & (coverage_comparison["Extreme_Coverage"] >= 70)
)

print("F4Z COVERAGE COMPARISON")
print("=" * 60)

print("Total features:", len(coverage_comparison))
print("Zero coverage in both:", int(zero_both.sum()))
print("90%+ coverage in both:", int(high_both.sum()))
print("70%+ coverage in both:", int(moderate_both.sum()))

print("\nCoverage by feature family:")

family_summary = (
    coverage_comparison.groupby("Family")
    .agg(
        Total_Features=("Feature", "count"),
        Average_Normal_Coverage=("Non_Extreme_Coverage", "mean"),
        Average_Extreme_Coverage=("Extreme_Coverage", "mean")
    )
    .reset_index()
)

display(family_summary.round(2))

Identify features with unusual coverage differences

In [ ]:
# Largest absolute differences
coverage_comparison["Absolute_Difference"] = (
    coverage_comparison["Coverage_Difference"].abs()
)

largest_differences = (
    coverage_comparison
    .sort_values("Absolute_Difference", ascending=False)
    .head(20)
)

print("TOP 20 FEATURES WITH LARGEST COVERAGE DIFFERENCES")
print("=" * 70)

display(
    largest_differences[
        [
            "Feature",
            "Family",
            "Non_Extreme_Coverage",
            "Extreme_Coverage",
            "Coverage_Difference"
        ]
    ].round(2)
)

Select features for investigation

In [ ]:
# Select representative features
selected_features = [
    "f4z__basis__basis_change_percentile__5m__value",
    "f4z__basis__basis_change_percentile__1d__value",
    "f4z__funding__funding_range_rz__1h__value"
]

# Add one zero-coverage feature
zero_features = coverage_comparison[
    (coverage_comparison["Non_Extreme_Coverage"] == 0)
    & (coverage_comparison["Extreme_Coverage"] == 0)
]["Feature"].tolist()

if zero_features:
    selected_features.append(zero_features[0])

print("SELECTED FEATURES")
print("=" * 70)

for feature in selected_features:
    print(feature)

Inspect invalid reasons efficiently

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
from collections import Counter

status_results = []

for dataset_name, pf in [
    ("NON_EXTREME", normal_pf),
    ("EXTREME", extreme_pf)
]:

    print(f"\nProcessing {dataset_name}...")

    for feature in selected_features:

        base = feature.removesuffix("__value")

        invalid_col = base + "__invalid_reason"
        status_col = base + "__scientific_status"

        available_cols = [
            col for col in [invalid_col, status_col]
            if col in pf.schema_arrow.names
        ]

        counters = {
            col: Counter()
            for col in available_cols
        }

        for batch in pf.iter_batches(
            batch_size=16384,
            columns=available_cols,
            use_threads=False
        ):

            for col in available_cols:
                array = batch.column(
                    batch.schema.get_field_index(col)
                )

                # Count unique status values in this small batch
                for value, count in Counter(
                    array.to_pylist()
                ).items():
                    counters[col][value] += count

        for col, counter in counters.items():
            for value, count in counter.items():

                status_results.append({
                    "Dataset": dataset_name,
                    "Feature": feature,
                    "Status Column": col,
                    "Status Value": str(value),
                    "Count": count
                })

status_df = pd.DataFrame(status_results)

print("\nINVALID REASON / SCIENTIFIC STATUS SUMMARY")

display(
    status_df.sort_values(
        ["Dataset", "Feature", "Count"],
        ascending=[True, True, False]
    )
)

Save inspection results

In [ ]:
import os
import json

RESULTS_DIR = "/content/drive/MyDrive/ScaleWave/Results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Save feature validity audit
validity_df.to_csv(
    os.path.join(RESULTS_DIR, "feature_coverage.csv"),
    index=False
)

# Save coverage comparison
coverage_comparison.to_csv(
    os.path.join(RESULTS_DIR, "coverage_comparison.csv"),
    index=False
)

# Save invalid-reason investigation
status_df.to_csv(
    os.path.join(RESULTS_DIR, "invalid_reason_summary.csv"),
    index=False
)

# Save return category summary
summary.to_csv(
    os.path.join(RESULTS_DIR, "return_distribution.csv"),
    index=False
)

# Save discovery/validation event counts
period_summary.to_csv(
    os.path.join(RESULTS_DIR, "research_period_summary.csv")
)

# Save episode summary
positive_episodes.to_csv(
    os.path.join(RESULTS_DIR, "positive_event_episodes.csv"),
    index=False
)

# Save the precise research boundaries
research_config = {
    "symbol": "BTCUSDT",
    "target": "signed_return",
    "horizon_minutes": 2880,
    "event_threshold_bps": 500,
    "discovery_end_exclusive": "2024-07-01T00:00:00Z",
    "validation_start_inclusive": "2024-07-03T00:00:00Z",
    "feature_prefix": "f4z__",
    "feature_suffix": "__value"
}

with open(
    os.path.join(RESULTS_DIR, "research_config.json"),
    "w"
) as f:
    json.dump(research_config, f, indent=4)

print("All inspection results saved successfully!")
print("Location:", RESULTS_DIR)